In [1]:
#Imports necessários

import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

In [2]:
#Configauração do Mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")

mlflow.set_experiment("Logistic_Regression")

mlflow.sklearn.autolog()

In [3]:
#Carregar o dataset
df = pd.read_csv("../DATASET/dataset_expandido.csv")

In [4]:
#Converter YearsCodePro para numérico

df["YearsCodePro_Num"] = df["YearsCodePro"]

df["YearsCodePro_Num"] = (
    df["YearsCodePro_Num"]
    .replace({
        "Less than 1 year":0,
        "More than 50 years":51,
        "Sem dado":0
    })
)

df["YearsCodePro_Num"] = pd.to_numeric(
    df["YearsCodePro_Num"],
    errors="coerce"
)

In [5]:
#Fazer a seleção das nossas features e do target


features = [
    "YearsCodePro_Num",
    "WorkExp",
    "Age_Code"
]

target = "JobSat_Class"

In [6]:
#Dataset final - junção das features e do target

df_log = df[
    features + [target]
].copy()

print(df_log.head())

print(df_log.info())

   YearsCodePro_Num  WorkExp  Age_Code JobSat_Class
0                 0      9.0         1        Medio
1                17     17.0         4        Medio
2                27      9.0         5        Medio
3                 0      9.0         2        Medio
4                 0      9.0         2        Medio
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 165437 entries, 0 to 165436
Data columns (total 4 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   YearsCodePro_Num  165437 non-null  int64  
 1   WorkExp           165437 non-null  float64
 2   Age_Code          165437 non-null  int64  
 3   JobSat_Class      165437 non-null  object 
dtypes: float64(1), int64(2), object(1)
memory usage: 5.0+ MB
None


In [7]:
#Definir que o X é as features e o y é o target

X = df_log[features]

y = df_log[target]

In [8]:
def run_logistic_experiment(
    run_name,
    X,
    y,
    random_state=42,
    test_size=0.2,
    use_half_data=False
):

    if use_half_data:

        X, _, y, _ = train_test_split(
            X,
            y,
            test_size=0.5,
            random_state=random_state
        )

    X_train, X_test, y_train, y_test = (
        train_test_split(
            X,
            y,
            test_size=test_size,
            random_state=random_state
        )
    )

    with mlflow.start_run(
        run_name=run_name
    ):

        model = LogisticRegression(
            max_iter=1000
        )

        model.fit(
            X_train,
            y_train
        )

        y_pred = model.predict(
            X_test
        )

        accuracy = accuracy_score(
            y_test,
            y_pred
        )

        precision = precision_score(
            y_test,
            y_pred,
            average="weighted"
        )

        recall = recall_score(
            y_test,
            y_pred,
            average="weighted"
        )

        f1 = f1_score(
            y_test,
            y_pred,
            average="weighted"
        )

        mlflow.log_metric(
            "accuracy",
            accuracy
        )

        mlflow.log_metric(
            "precision",
            precision
        )

        mlflow.log_metric(
            "recall",
            recall
        )

        mlflow.log_metric(
            "f1",
            f1
        )

        print(run_name)

        print(
            "Accuracy:",
            accuracy
        )

        print(
            "Precision:",
            precision
        )

        print(
            "Recall:",
            recall
        )

        print(
            "F1:",
            f1
        )

        print("-"*40)

In [10]:
# Experiment 1 - Baseline

run_logistic_experiment(
    run_name=
    "Experiment_1_Baseline",
    X=X,
    y=y,
    random_state=42,
    test_size=0.2
)

2026/05/21 21:51:51 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/05/21 21:51:51 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c

Experiment_1_Baseline
Accuracy: 0.7167553191489362
Precision: 0.6278312893414623
Recall: 0.7167553191489362
F1: 0.6185433449717248
----------------------------------------


In [11]:
# Experiment 2 - Change Randomness

run_logistic_experiment(
    run_name=
    "Experiment_2_Change_Randomness",
    X=X,
    y=y,
    random_state=10,
    test_size=0.2
)

2026/05/21 21:52:10 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/21 21:52:11 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Lo

Experiment_2_Change_Randomness
Accuracy: 0.7197473404255319
Precision: 0.6338892292299987
Recall: 0.7197473404255319
F1: 0.6228793878649249
----------------------------------------


In [12]:
# Experiment 3 - Larger Test Set

run_logistic_experiment(
    run_name=
    "Experiment_3_Larger_Test_Set",
    X=X,
    y=y,
    random_state=42,
    test_size=0.4
)

2026/05/21 21:52:18 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/21 21:52:19 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Lo

Experiment_3_Larger_Test_Set
Accuracy: 0.7169172648281072
Precision: 0.628465532427614
Recall: 0.7169172648281072
F1: 0.6186026906248328
----------------------------------------


In [13]:
# Experiment 4 - Smaller Test Set

run_logistic_experiment(
    run_name=
    "Experiment_4_Smaller_Test_Set",
    X=X,
    y=y,
    random_state=42,
    test_size=0.1
)

2026/05/21 21:52:26 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/21 21:52:27 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Lo

Experiment_4_Smaller_Test_Set
Accuracy: 0.713914410058027
Precision: 0.6258056564327569
Recall: 0.713914410058027
F1: 0.6151821502746614
----------------------------------------


In [14]:
# Experiment 5 - Remove One Features

X_exp5 = df_log.drop(columns=["Age_Code",target])

run_logistic_experiment(
    run_name=
    "Experiment_5_Remove_One_Feature",
    X=X_exp5,
    y=y,
    random_state=42,
    test_size=0.2
)

2026/05/21 21:52:35 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/21 21:52:36 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Lo

Experiment_5_Remove_One_Feature
Accuracy: 0.7164833172147002
Precision: 0.6270211753159424
Recall: 0.7164833172147002
F1: 0.6177807256601378
----------------------------------------


In [15]:
# Experiment 6 - Remove Two Features

X_exp6 = df_log.drop(columns=["Age_Code","WorkExp",target])

run_logistic_experiment(
    run_name=
    "Experiment_6_Remove_Two_Features",
    X=X_exp6,
    y=y,
    random_state=42,
    test_size=0.2
)

2026/05/21 21:52:42 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/21 21:52:43 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Lo

Experiment_6_Remove_Two_Features
Accuracy: 0.7041827852998066
Precision: 0.5575877208048704
Recall: 0.7041827852998066
F1: 0.591142948085843
----------------------------------------


In [16]:
# Experiment 7 - Half Dataset

run_logistic_experiment(
    run_name=
    "Experiment_7_Reduce_Dataset_Size",
    X=X,
    y=y,
    random_state=42,
    test_size=0.2,
    use_half_data=True
)

2026/05/21 21:52:49 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/21 21:52:50 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Lo

Experiment_7_Reduce_Dataset_Size
Accuracy: 0.7168762088974855
Precision: 0.6213394514892007
Recall: 0.7168762088974855
F1: 0.6197235985356087
----------------------------------------


In [17]:
# Experiment 8 - Combine Changes

run_logistic_experiment(
    run_name=
    "Experiment_8_Combine_Changes",
    X=X,
    y=y,
    random_state=10,
    test_size=0.4
)

2026/05/21 21:52:55 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/21 21:52:56 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Lo

Experiment_8_Combine_Changes
Accuracy: 0.7203777861730261
Precision: 0.6307284944660487
Recall: 0.7203777861730261
F1: 0.6245324565039274
----------------------------------------


### Introdução

Depois de efetuadas as experiências conseguimos verificar que os resultados foram muito superiores, relativamnete à Linear Regression. Neste caso da Logistic Regression, o modelo conseguiu alcançar valores da Accuracy próximos de 72%, indicando uma capacidade moderada para classificar o nível de satisfação profissional (JobSat_Class).

Verificamos ainda que de uma forma geral, todas as experiências tiveram valores relativamente próximos, o que nos leva a crer que o modelo é estável e pouco sensível a pequenas alterações nos dados.

### ANÁLISE DE EXPERIÊNCIAS

# Experiência 1 - Baseline

Através dos resultados obtidos nesta experiência, verificamos que:

    .Accuracy ≈ 71.7%
    .F1 ≈ 0.619

Com estes resultados conseguimos concluir qye as variáveis yearsCodePro_Num, WorkExp e Age_Code possuem uma certa capacidade para explicar parcialmente a satisfação profissional dos developers.
De forma comparativa com a Regressão Linear, este desempenho foi significativamente superior.

# Experiência 2 - Change Randomness

Ao alterar o valor do random_state para 10 houve uma pequena melhoria. A nossa accuracy passou para 71.97%. Este comportamento mostra que o modelo possui estabilidade relativamnete à divisão do dataset.

Concluimos portanto que a performance da Logistic Regression não depende fortemente da divisão dos dados.

# Experiência 3 - Larger Test Set

A experiência 3 foi modificar o valor do test_size. Neste caso a alteração foi de 0.2 para 0.4, o que manteve o desempenho do modelo muito próximo ao desempenho da experiência Baseline. Ou seja, o modelo generaliza bem, mesmo utilizando menos dados de treino.

# Experiência 4 - Smaller Test Set

Nesta experiência, ao contrário da anterior (experiência 3) diminuimos o valor do test_size para 0.1 e com isso surgiu uma redução do desempenho do nosso modelo, atingindo uam accuracy de 71.39%. 

Ou seja, apesar de o número de dados de treino ter aumentado, não existiu nenhum tipo de melhoria em nenhum aspeto. 

# Experiência 5 - Remove One Feature

Nesta experiência o objetivo era remover uma das features, no caso removemos a feature Age_Code e isso provocou diferenças negativas (quase nulas), ou seja, a performance praticamente não se alterou. E com isso concluimos que a idade tem baixa influência na previsão da satisfação profissional.

# Experiência 6 - Remove Two Features

Esta experiência, foi a que obteve piores resultados. Terminou com uma accuracy de 70.41% e um F1 de 0.59. 

Portanto, ao remover duas features (Age_Code e WorkExp), o modelo perdeu informação relevante. Os resultados mostram que a experiência profissional possui influência na satisfação.

Porque na experiência anterior ao remover Age_Code não houve quase alterações e nesta ao remover a mesma feature + a WorkExpr os resultados pioram significativamente, o que nos leva a crer que WorkExp possui influência.

# Experiência 7 - Reduce Dataset Size

Nesta experiência os resultados foram bastante parecidos. Esta experiência perde apenas no quesito 'Precision'.
Verificamos então que o dataset muito provavelmente possui redundância, porque teve os resultados muito parecidos e apenas com metade do dataset.

# Experiência 8 - Combine Changes

Esta foi a melhor experiência de todas, porque foi a que ealcançou o maior valor da accuracy (72.03%) e também de F1 (0.62). A combinação, da alteração entre o random_state (diminuição) e do test_size (aumento), produziu os melhores resultados e o melhro equilibrio entre generalização e desempenho. 

### Conclusão

Para concluir a análise deste modelo, conseguimos retirar que a regressão logistica demonstrou ser adequada para a classificação do nível de satisfação profissional. Comparativamente com a Regressão Linear, os resultados foram significativamente superiores.

Além disso, o modelo mostrou elevada estabilidade entre experiências, indicando boa capacidade de generalização.

# Hyperparameter Optimization - Logistic Regression

In [9]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression

import optuna

In [12]:
target = "JobSat_Class"

features = [

    "YearsCodePro_Num",

    "WorkExp",

    "Age_Code"

]

X = df[features]

y = df[target]

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.2,

    random_state=42

)

In [13]:
mlflow.sklearn.autolog(

    disable=True

)

In [14]:
param_grid = {

    "C":[

        0.001,

        0.01,

        0.1,

        1,

        10

    ],

    "penalty":[

        "l2"

    ],

    "solver":[

        "lbfgs"

    ]

}

grid = GridSearchCV(

    LogisticRegression(

        max_iter=1000

    ),

    param_grid,

    cv=10,

    scoring="accuracy"

)

grid.fit(

    X_train,

    y_train

)

print(

    "Best Parameters:",

    grid.best_params_

)

print(

    "Best Score:",

    grid.best_score_

)

c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was depr

Best Parameters: {'C': 0.01, 'penalty': 'l2', 'solver': 'lbfgs'}
Best Score: 0.71852448607635


In [15]:
with mlflow.start_run(

    run_name="GridSearch_Logistic"

):

    mlflow.log_param(

        "method",

        "GridSearch"

    )

    mlflow.log_params(

        grid.best_params_

    )

    mlflow.log_metric(

        "best_score",

        grid.best_score_

    )

2026/05/25 23:01:05 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



In [16]:
def objective(

    trial

):

    C = trial.suggest_categorical(

        "C",

        [

            0.001,

            0.01,

            0.1,

            1,

            10

        ]

    )

    model = LogisticRegression(

        C=C,

        max_iter=1000

    )

    scores = cross_val_score(

        model,

        X_train,

        y_train,

        cv=10,

        scoring="accuracy"

    )

    return scores.mean()

In [17]:
study = optuna.create_study(

    direction="maximize"

)

study.optimize(

    objective,

    n_trials=5

)

print(

    "Best Parameters:",

    study.best_params

)

print(

    "Best Score:",

    study.best_value
)

[I 2026-05-25 23:01:33,783] A new study created in memory with name: no-name-f51e9f5f-7802-4973-a01f-7e2b6b4dfc9e
[I 2026-05-25 23:01:51,150] Trial 0 finished with value: 0.71852448607635 and parameters: {'C': 1}. Best is trial 0 with value: 0.71852448607635.
[I 2026-05-25 23:02:10,947] Trial 1 finished with value: 0.71852448607635 and parameters: {'C': 1}. Best is trial 0 with value: 0.71852448607635.
[I 2026-05-25 23:02:23,063] Trial 2 finished with value: 0.71852448607635 and parameters: {'C': 0.1}. Best is trial 0 with value: 0.71852448607635.
[I 2026-05-25 23:02:34,355] Trial 3 finished with value: 0.7185169303528895 and parameters: {'C': 10}. Best is trial 0 with value: 0.71852448607635.
[I 2026-05-25 23:02:45,538] Trial 4 finished with value: 0.71852448607635 and parameters: {'C': 0.1}. Best is trial 0 with value: 0.71852448607635.


Best Parameters: {'C': 1}
Best Score: 0.71852448607635


In [18]:
with mlflow.start_run(

    run_name="Optuna_Logistic"

):

    mlflow.log_param(

        "method",

        "Optuna"

    )

    mlflow.log_params(

        study.best_params

    )

    mlflow.log_metric(

        "best_score",

        study.best_value
    )